# Quick Web Crawl Tester

Nhập danh sách URL và chạy để kiểm tra có tải được HTML/text hay không (không cần khởi động toàn bộ pipeline).


In [ ]:
import os
import asyncio
from typing import Sequence

import aiohttp

try:
    from bs4 import BeautifulSoup
except ImportError as exc:  # pragma: no cover
    raise ImportError("Vui lòng cài beautifulsoup4: pip install beautifulsoup4") from exc

os.environ.setdefault("WEB_SEARCH_SSL", "False")

DEFAULT_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
}


async def _fetch(url: str, ssl: bool = False) -> str:
    timeout = aiohttp.ClientTimeout(total=60)
    connector = aiohttp.TCPConnector(ssl=ssl)
    async with aiohttp.ClientSession(timeout=timeout, connector=connector) as session:
        async with session.get(url, headers=DEFAULT_HEADERS) as response:
            response.raise_for_status()
            return await response.text()


def _extract_text(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    text = "\n".join(chunk.strip() for chunk in soup.stripped_strings if chunk.strip())
    return text


async def crawl_urls(urls: Sequence[str], ssl: bool = False) -> list[dict[str, str]]:
    results = []
    for url in urls:
        try:
            html = await _fetch(url, ssl=ssl)
            text = _extract_text(html)
            results.append({"url": url, "html_len": len(html), "text": text})
        except Exception as exc:
            results.append({"url": url, "error": str(exc)})
    return results



In [ ]:
URLS = [
    "https://www.uet.vnu.edu.vn/loi-nguoc-dong-thanh-thu-khoa-dai-hoc/",
    # Thêm bớt URL tại đây
]

SSL_FLAG = os.environ.get("WEB_SEARCH_SSL", "False").lower() in ("true", "1")
print(f"Testing {len(URLS)} URL(s) | ssl={SSL_FLAG}")


In [ ]:
results = asyncio.run(crawl_urls(URLS, ssl=SSL_FLAG))

for idx, item in enumerate(results, start=1):
    print("=" * 80)
    print(f"Result {idx}: {item['url']}")
    if "error" in item:
        print(f"❌ Error: {item['error']}")
    else:
        snippet = item["text"][:1000]
        print(f"HTML length: {item['html_len']:,}")
        print(f"Text preview ({len(snippet)} chars):\n{snippet}")
